# Creating an agent with MCP

The [Model Context Protocol](https://www.anthropic.com/news/model-context-protocol) (MCP) introduced by Anthropic has proven to be a popular method for providing an AI agent with access to a variety of tools. [This Huggingface blog post ](https://huggingface.co/blog/Kseniase/mcp) has a nice explanation of MCP.  In this tutorial, we'll build an agent that is able to leverage MCP server provided tools.

Note: because this tutorial relies upon advanced stdio/stderr communication using the MCP Server, it cannot be run on Google Colab.

## Install Dependencies

any-agent uses the python asyncio module to support async functionality. When running in Jupyter notebooks, this means we need to enable the use of nested event loops. We'll install any-agent and enable this below using nest_asyncio.

In [1]:
#!uv pip install 'any-agent'

import nest_asyncio

nest_asyncio.apply()

## Configure the Agent

Now it's time to configure the agent! At this stage you have a few choices:

### Pick the framework

We support a variety of underlying agent frameworks (OpenAI, Smolagents, Langchain, TinyAgent, etc), which all have their own particular agentic AI implementations. For this tutorial's simple use case, any of the frameworks should work just fine, but any-agent makes it easy to try out a different framework later, if we so choose. For this example, we will use the [TinyAgent](frameworks/tinyagent.md) framework.  

### Pick an LLM

Regardless of which agent framework you choose, each framework supports LiteLLM, which is a proxy that allows us to use whichever LLM inside the framework, hosted on by any provider. For example, we could use a local model via llama.cpp or llamafile, a google hosted gemini model, or a AWS bedrock hosted Llama model. For this example, let's use Mistral AI's mistral/mistral-small-latest.

### Pick which tools to use

 In this example, we'll add a few MCP servers that we host locally, which means we'll use a Stdio MCP server. If an MCP Server is already running and hosted elsewhere, you can use an SSE connection to access it. You can browse some of the officially supported MCP servers [here](https://github.com/modelcontextprotocol/servers/tree/main?tab=readme-ov-file).

 Lets give use two MCP servers: 
 
 * [Time](https://github.com/modelcontextprotocol/servers/tree/main/src/time): so the agent can know what time/day it is.
 * [Airbnb](https://github.com/openbnb-org/mcp-server-airbnb): so the agent can browse airbnb listings

 I will also add a custom send_message tool, that way it can ask us additional questions before getting its final answer!
  

In [7]:
import os
from getpass import getpass
from any_agent import AgentConfig, AnyAgent
from any_agent.config import MCPStdio

if "GEMINI_API_KEY" not in os.environ:
    print("GEMINI_API_KEY not found in environment!")
    api_key = getpass("Please enter your GEMINI_API_KEY: ")
    os.environ["GEMINI_API_KEY"] = api_key
    print("GEMINI_API_KEY set for this session!")
else:
    print("GEMINI_API_KEY found in environment.")

# Placeholder for send_message tool
def send_message(message: str) -> str:
    """Display a message to the user and wait for their response."""
    return input(message + " ")

# Initialize JobScoutAgent (tools will be added below)
agent = AnyAgent.create(
    "TINYAGENT", 
    AgentConfig(
        model_id="gemini/gemini-2.5-flash",
        tools=[send_message]
    )
)


GEMINI_API_KEY found in environment.


In [5]:
!uv pip install pdfminer.six

Using Python 3.12.7 environment at: /Users/koware/Desktop/SECOM_HTWG2-Kwaku/Agent_AI/any-agent/.venv
Resolved 5 packages in 246ms                                         
Prepared 2 packages in 292ms                                             
Installed 2 packages in 6ms06                               
 + cryptography==45.0.5
 + pdfminer-six==20250506


In [8]:
from typing import Annotated
from pdfminer.high_level import extract_text

def parse_resume(resume_path: Annotated[str, "Path to a resume PDF file."]) -> str:
    """
    Extract plain text from a resume PDF file.

    Args:
        resume_path: Path to the PDF resume file.

    Returns:
        str: Extracted plain text from the resume.
    """
    try:
        text = extract_text(resume_path)
        return text.strip()
    except Exception as e:
        return f"Error parsing resume: {str(e)}"


In [10]:
agent = AnyAgent.create(
    "TINYAGENT",
    AgentConfig(
        model_id="gemini/gemini-2.5-flash",
        tools=[
            send_message,
            parse_resume,  # now included
        ]
    )
)


In [11]:
from typing import Annotated, List, Dict

def search_jobs(
    query: Annotated[str, "What kind of job is the user looking for?"],
    location: Annotated[str, "Preferred job location (remote, city, country)."],
    num_results: Annotated[int, "Number of job results to return."] = 5
) -> List[Dict]:
    """
    Simulate a job search based on query and location. Returns mock job listings.

    Args:
        query: Job title or keywords
        location: Preferred location (e.g., 'remote', 'Berlin')
        num_results: Number of jobs to return

    Returns:
        A list of job postings, each as a dict with title, company, description, and URL
    """
    mock_jobs = [
        {
            "title": "Data Analyst",
            "company": "TechCorp",
            "location": location,
            "description": "Analyze business data and build dashboards using Python and SQL.",
            "url": "https://jobs.example.com/data-analyst"
        },
        {
            "title": "Machine Learning Engineer",
            "company": "InnovateAI",
            "location": location,
            "description": "Develop and deploy ML models in production using PyTorch and TensorFlow.",
            "url": "https://jobs.example.com/ml-engineer"
        },
        {
            "title": "Backend Developer",
            "company": "CodeBase",
            "location": location,
            "description": "Build REST APIs with Node.js and Express for scalable systems.",
            "url": "https://jobs.example.com/backend-developer"
        },
        {
            "title": "Healthcare Data Scientist",
            "company": "HealthAI",
            "location": location,
            "description": "Use statistical modeling to analyze clinical data and improve patient outcomes.",
            "url": "https://jobs.example.com/health-data-scientist"
        },
        {
            "title": "Product Manager - AI Tools",
            "company": "AIBridge",
            "location": location,
            "description": "Manage AI product lifecycle from ideation to launch, focusing on LLM features.",
            "url": "https://jobs.example.com/product-manager"
        },
    ]
    return mock_jobs[:num_results]


In [12]:
agent = AnyAgent.create(
    "TINYAGENT",
    AgentConfig(
        model_id="gemini/gemini-2.5-flash",
        tools=[
            send_message,
            parse_resume,
            search_jobs,  # ✅ newly added
        ]
    )
)


In [14]:
!uv pip install scikit-learn

Using Python 3.12.7 environment at: /Users/koware/Desktop/SECOM_HTWG2-Kwaku/Agent_AI/any-agent/.venv
Resolved 5 packages in 303ms                                         
Prepared 3 packages in 1.23s                                             
Installed 5 packages in 66ms                                
 + joblib==1.5.1
 + numpy==2.3.2
 + scikit-learn==1.7.1
 + scipy==1.16.1
 + threadpoolctl==3.6.0


In [15]:
from typing import Annotated
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def match_jobs(
    resume_text: Annotated[str, "Extracted plain text from the user's resume."],
    jobs: Annotated[List[Dict], "List of job postings to compare against."]
) -> List[Dict]:
    """
    Match resume to job descriptions using cosine similarity.

    Args:
        resume_text: Text extracted from the user's resume.
        jobs: List of job dicts with 'description' fields.

    Returns:
        List of top job matches sorted by similarity score (desc).
    """
    job_descriptions = [job["description"] for job in jobs]
    titles = [job["title"] for job in jobs]

    # Combine resume + job descriptions for vectorization
    documents = [resume_text] + job_descriptions
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(documents)

    # Compute cosine similarity (resume vs each job)
    similarities = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()

    # Attach similarity scores to each job
    for i, score in enumerate(similarities):
        jobs[i]["match_score"] = round(float(score), 3)

    # Sort jobs by highest match score
    sorted_jobs = sorted(jobs, key=lambda x: x["match_score"], reverse=True)
    return sorted_jobs


In [16]:
agent = AnyAgent.create(
    "TINYAGENT",
    AgentConfig(
        model_id="gemini/gemini-2.5-flash",
        tools=[
            send_message,
            parse_resume,
            search_jobs,
            match_jobs,  # ✅ newly added
        ]
    )
)


In [17]:
# Sample resume text (you can replace this with actual output from parse_resume)
sample_resume = """
Experienced data analyst with proficiency in Python, SQL, and data visualization.
Built dashboards using Power BI and automated reports. Experience with A/B testing and machine learning models.
"""

# Simulate job search results
jobs = search_jobs("data", "Remote", 5)

# Match the resume against the jobs
matched = match_jobs(sample_resume, jobs)

# Display top matches
for job in matched:
    print(f"{job['title']} at {job['company']}")
    print(f"Score: {job['match_score']}")
    print(f"Description: {job['description']}")
    print(f"Link: {job['url']}")
    print("-" * 40)


Data Analyst at TechCorp
Score: 0.352
Description: Analyze business data and build dashboards using Python and SQL.
Link: https://jobs.example.com/data-analyst
----------------------------------------
Machine Learning Engineer at InnovateAI
Score: 0.101
Description: Develop and deploy ML models in production using PyTorch and TensorFlow.
Link: https://jobs.example.com/ml-engineer
----------------------------------------
Healthcare Data Scientist at HealthAI
Score: 0.079
Description: Use statistical modeling to analyze clinical data and improve patient outcomes.
Link: https://jobs.example.com/health-data-scientist
----------------------------------------
Backend Developer at CodeBase
Score: 0.0
Description: Build REST APIs with Node.js and Express for scalable systems.
Link: https://jobs.example.com/backend-developer
----------------------------------------
Product Manager - AI Tools at AIBridge
Score: 0.0
Description: Manage AI product lifecycle from ideation to launch, focusing on LLM

In [7]:
from any_agent import AgentConfig, AnyAgent
from any_agent.config import MCPStdio

# This MCP Tool relies upon uvx https://docs.astral.sh/uv/getting-started/installation/
time_tool = MCPStdio(
    command="uvx",
    args=["mcp-server-time", "--local-timezone=America/New_York"],
    tools=[
        "get_current_time",
    ],
    client_session_timeout_seconds=30,
)

# This MCP tool relies upon npx https://docs.npmjs.com/cli/v8/commands/npx which comes standard with npm
airbnb_tool = MCPStdio(
    command="npx",
    args=["-y", "@openbnb/mcp-server-airbnb", "--ignore-robots-txt"],
    client_session_timeout_seconds=30,
)


# This is a custom tool that we will provide to the agent. For the agent to use the tool, we must provide a docstring
# and also have proper python typing for input and output parameters
def send_message(message: str) -> str:
    """Display a message to the user and wait for their response.

    Args:
        message: str
            The message to be displayed to the user.

    Returns:
        str: The response from the user.

    """
    if os.environ.get("IN_PYTEST") == "1":
        return "2 people, next weekend, low budget. Do not ask for any more information or confirmation."
    return input(message + " ")


agent = AnyAgent.create(
    "tinyagent",  # See all options in https://mozilla-ai.github.io/any-agent/
    AgentConfig(
        model_id="gemini/gemini-2.5-flash",
        tools=[airbnb_tool, time_tool, send_message],
    ),
)

## Run the Agent

Now we've configured our agent, so it's time to run it! Since it has access to airbnb listings as well as the current time, it's a perfect fit for helping me find a nice airbnb for the weekend.


In [8]:
prompt = """
I am looking to book an airbnb next weekend near Ohiopyle, PA. Can you help me plan this? Figure out the time, then ask me some questions and lets figure this out together. Once I tell you what you need to know, find and return some options for me.
"""

agent_trace = agent.run(prompt)

╭─────────────────────────────────────── CALL_LLM: gemini/gemini-2.5-flash ───────────────────────────────────────╮
│ ╭─ INPUT ─────────────────────────────────────────────────────────────────────────────────────────────────────╮ │
│ │ [                                                                                                           │ │
│ │   {                                                                                                         │ │
│ │     "role": "system",                                                                                       │ │
│ │     "content": "You are an agent - please keep going until the user's query is completely resolved, before  │ │
│ │   },                                                                                                        │ │
│ │   {                                                                                                         │ │
│ │     "role": "user",                                                                                         │ │
│ │     "content": "\nI am looking to book an airbnb next weekend near Ohiopyle, PA. Can you help me plan this? │ │
│ │   }                                                                                                         │ │
│ │ ]                                                                                                           │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
│ ╭─ OUTPUT ────────────────────────────────────────────────────────────────────────────────────────────────────╮ │
│ │ To help you plan your Airbnb, I'll need to know your current timezone to figure out "next weekend."         │ │
│ │                                                                                                             │ │
│ │ What is your current timezone (e.g., 'America/New_York', 'Europe/London')?                                  │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
│ ╭─ USAGE ─────────────────────────────────────────────────────────────────────────────────────────────────────╮ │
│ │ {                                                                                                           │ │
│ │   "input_tokens": 876,                                                                                      │ │
│ │   "output_tokens": 236,                                                                                     │ │
│ │   "input_cost": 0.0002628,                                                                                  │ │
│ │   "output_cost": 0.00059                                                                                    │ │
│ │ }                                                                                                           │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## View the results 

The `agent.run` method returns an AgentTrace object, which has a few convenient attributes for displaying some interesting information about the run.

In [9]:
print(agent_trace.final_output)  # Final answer
print(f"Duration: {agent_trace.duration.total_seconds():.2f} seconds")
print(f"Total Tokens: {agent_trace.tokens.total_tokens:,}")
print(f"Total Cost (USD): {agent_trace.cost.total_cost:.6f}")

To help you plan your Airbnb, I'll need to know your current timezone to figure out "next weekend."

What is your current timezone (e.g., 'America/New_York', 'Europe/London')?
Duration: 2.27 seconds
Total Tokens: 1,112
Total Cost (USD): 0.000853
